# 07 — Segmentação de clientes (clusterização)
**Hackathon Databricks · Terra Signal · Grupo 4**

**Equipe:** André Ferreira, Antonio Menezes, Júlia Vale e Lucas Menghi

**O que este notebook faz:** agrupa os clientes em **segmentos** com comportamento parecido e cruza cada segmento com o
**risco de churn** do modelo e com o **valor** do cliente. O modelo responde **quem** vai sair; a segmentação responde
**que tipo de cliente** é, e daí **que oferta** faz sentido.

**O notebook é organizado em perguntas de negócio.** Cada etapa abre com a pergunta, mostra a teoria e o passo a
passo, e fecha com a **resposta**, escrita a partir dos próprios números. No fim, uma tabela junta todas as respostas.

| # | Pergunta de negócio | Etapa |
|---|---|---|
| P1 | Com que informação descrevemos o cliente, sem usar o churn? | 1. Variáveis de comportamento |
| P2 | Quantos grupos de clientes fazem sentido para o negócio usar? | 2. K-means e silhueta |
| P3 | Que tipos de cliente temos, e o que caracteriza cada um? | 3. Perfil dos segmentos |
| P4 | Onde o churn e a receita se concentram? | 3. Churn e receita por segmento |
| P5 | Entre os clientes que vamos pontuar, quem priorizar? | 4. Risco × valor |
| P6 | Quanto da receita está em risco, e em que segmento? | 4. Receita em risco |
| P7 | Que oferta fazer a cada segmento? | 4. Ação sugerida |

**De onde vem o risco:** do modelo do grupo (notebook 06), pela tabela `gold_churn_inferencia_scored`:
a probabilidade (`Probabilidade_Churn`) e a previsão Sim/Não no corte definido no 06 (`Churn_Previsto`). Se a tabela não
existir, o notebook usa uma regressão logística provisória com a seleção da conferência do 05 (seção 5).

In [ ]:
%run ./_comum

## Glossário deste notebook

| Termo | Por extenso | O que é, em uma frase | Onde entender melhor |
|---|---|---|---|
| **K-means** | K-médias | Algoritmo que divide os clientes em K grupos, cada um em volta de um "cliente médio" (centroide) | Seção 2 |
| **Centroide** | — | O perfil médio de um segmento: a média de cada variável dentro dele | Seção 2 |
| **Silhueta** | Coeficiente de silhueta | O quanto cada cliente está mais perto do próprio segmento do que do vizinho (−1 a 1) | Seção 2 |
| **ABT** | *Analytical Base Table* (tabela analítica de base) | A tabela que o modelo lê, na camada gold | 04: seção 2 |
| **AUC** | *Area Under the Curve* (área sob a curva ROC) | A chance de o modelo dar nota de risco maior a quem saiu do que a quem ficou | 05: seção 5 |
| **D20** | Decisão do checklist | Segmentação de clientes (este notebook) | Checklist final |

## 1. Variáveis de comportamento

> **P1 — Com que informação descrevemos o cliente, sem usar o churn?**

### Teoria: o que é segmentar, e por que sem o churn

**Segmentar** é dividir a carteira em grupos de clientes parecidos entre si e diferentes dos outros grupos. É uma
técnica **não supervisionada**: não existe uma resposta certa para aprender, como o churn no modelo. O algoritmo só
olha as características e procura grupos naturais.

**Por que deixar o churn de fora:** queremos grupos que descrevam **como o cliente se relaciona com a empresa**
(contrato, serviços, pagamento, tempo de casa), e não grupos que repitam o modelo. Depois, medimos o churn de cada
grupo: se os grupos tiverem churn muito diferente, a segmentação é útil para o negócio **e** coerente com o modelo.

**Variáveis usadas** (da ABT, gold):

| Tema | Variáveis |
|---|---|
| Relacionamento | `tenure` (meses de casa), `Contract` (mensal, anual, dois anos) |
| Pagamento | pagamento automático, cheque eletrônico, fatura digital |
| Pacote | `InternetService` (DSL, fibra, sem internet), os 4 serviços de proteção (`_bin`), nº de streamings |
| Perfil | idoso, tem família (parceiro ou dependentes) |

**Ficam de fora:** o `Churn` (o que queremos explicar depois), o texto (vazamento, D9), a renda (sem informação,
D12), o sexo (variável sensível, D15) e a mensalidade, que entra depois como **valor** do cliente (seção 4).

### Teoria: por que colocar tudo na mesma escala

O K-means mede a **distância** entre clientes. Se `tenure` vai de 0 a 72 e as outras variáveis de 0 a 1, o tempo de
casa dominaria a distância sozinho. Por isso tudo vai para a escala de 0 a 1: `tenure` ÷ 72, nº de streamings ÷ 2, e
as categorias viram colunas 0/1.

In [ ]:
# 1.1 Carrega a gold e monta a matriz de comportamento (tudo entre 0 e 1)
assert "FQ" in globals(), "rode antes a célula %run ./_comum"
treino = spark.table(f"{FQ}.gold_churn_treino").toPandas()
inferencia = spark.table(f"{FQ}.gold_churn_inferencia").toPandas()
assert (len(treino), len(inferencia)) == (5634, 1409), "a gold não está como esperado: rodar o notebook 04"

def comportamento(d: pd.DataFrame) -> pd.DataFrame:
    X = pd.DataFrame(index=d.index)
    X["meses de casa"] = d["tenure"] / 72
    for cat, nome in [("Month-to-month", "contrato mensal"), ("One year", "contrato anual"), ("Two year", "contrato de dois anos")]:
        X[nome] = d["Contract"].eq(cat).astype(float)
    for cat, nome in [("DSL", "internet DSL"), ("Fiber optic", "internet fibra"), ("No", "sem internet")]:
        X[nome] = d["InternetService"].eq(cat).astype(float)
    X["pagamento automático"] = d["pagamento_automatico"]
    X["cheque eletrônico"] = d["is_electronic_check"]
    X["fatura digital"] = d["PaperlessBilling"].eq("Yes").astype(float)
    for c, nome in [("OnlineSecurity", "segurança online"), ("OnlineBackup", "backup online"),
                    ("DeviceProtection", "proteção de aparelho"), ("TechSupport", "suporte técnico")]:
        X[nome] = d[f"{c}_bin"]
    X["streamings (0 a 2)"] = d["n_streaming"] / 2
    X["idoso"] = d["SeniorCitizen"]
    X["tem família"] = d["has_family"]
    return X.astype(float)

X_treino, X_inf = comportamento(treino), comportamento(inferencia)
assert X_treino.isna().sum().sum() == 0 and X_treino.min().min() >= 0 and X_treino.max().max() <= 1
print(f"Matriz de comportamento: {virg(len(X_treino), 0)} clientes × {X_treino.shape[1]} variáveis, todas entre 0 e 1")
RESPOSTAS = {}
RESPOSTAS["P1"] = (f"Com {X_treino.shape[1]} características de relacionamento, pagamento, pacote e perfil (contrato, meses de casa, "
                   "tipo de internet, serviços de proteção, forma de pagamento, idoso, família), sem churn, texto, renda e sexo")
print("\nResposta P1:", RESPOSTAS["P1"])

## 2. Encontrar os segmentos (K-means)

> **P2 — Quantos grupos de clientes fazem sentido para o negócio usar?**

### Teoria: como o K-means funciona

1. Escolho **K**, o número de segmentos.
2. O algoritmo sorteia K pontos de partida e coloca cada cliente no grupo do ponto **mais próximo**.
3. Recalcula o **centroide** de cada grupo (o cliente médio) e redistribui os clientes.
4. Repete até os grupos pararem de mudar.

Uma analogia: espalhar K mesas num salão e cada pessoa sentar na mais próxima; depois mover cada mesa para o centro de
quem sentou nela e repetir, até ninguém mais trocar de mesa.

### Teoria: como escolher K (silhueta)

A **silhueta** compara, para cada cliente, a distância média até o próprio grupo (*a*) e até o grupo vizinho mais
próximo (*b*): silhueta = (*b* − *a*) ÷ o maior dos dois. Vai de −1 a 1: perto de 1, o cliente está bem no seu grupo;
perto de 0, está na fronteira. A média de todos os clientes mede a qualidade da divisão.

**O critério não é só o número:** com variáveis 0/1, a silhueta costuma ficar baixa (0,2 é comum), porque muitos
clientes ficam "entre" grupos. Por isso escolho K pela **silhueta** e pela **utilidade para o negócio**: entre 4 e 6
segmentos que tenham nome, perfil claro e churn diferente entre si.

In [ ]:
# 2.1 Testa de 3 a 8 segmentos: silhueta e diferença de churn entre os segmentos
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

y = treino["Churn"]
testes_k = []
for k in range(3, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_treino)
    churn_grupos = y.groupby(km.labels_).mean()
    testes_k.append({"K": k, "silhueta": silhouette_score(X_treino, km.labels_, sample_size=3000, random_state=42),
                     "churn_min_%": 100 * churn_grupos.min(), "churn_max_%": 100 * churn_grupos.max(),
                     "menor_segmento": int(np.bincount(km.labels_).min())})
testes_k = pd.DataFrame(testes_k)
K = int(testes_k.loc[testes_k["K"].between(4, 6)].sort_values("silhueta", ascending=False).iloc[0]["K"])
display(testes_k.round(3))
print(f"K escolhido: {K} (maior silhueta entre 4 e 6 segmentos)")
lin = testes_k.set_index("K").loc[K]
RESPOSTAS["P2"] = (f"{K} segmentos: a divisão mais nítida entre as opções úteis (silhueta {virg(lin['silhueta'], 2)}), "
                   f"com churn de {virg(lin['churn_min_%'], 0)}% a {virg(lin['churn_max_%'], 0)}% entre eles")
print("Resposta P2:", RESPOSTAS["P2"])

fig, ax = plt.subplots(figsize=(10, 3.6))
cores_k = [DESTAQUE if k == K else (COR if 4 <= k <= 6 else CINZA) for k in testes_k["K"]]
ax.bar(testes_k["K"].astype(str), testes_k["silhueta"], color=cores_k, width=0.6)
for x0, (_, r) in enumerate(testes_k.iterrows()):
    ax.text(x0, r["silhueta"] + 0.003, f"{virg(r['silhueta'], 3)}\nchurn {virg(r['churn_min_%'], 0)}% a {virg(r['churn_max_%'], 0)}%",
            ha="center", fontsize=8.5, color=TEXTO)
ax.set_ylim(0, testes_k["silhueta"].max() * 1.35)
ax.set_xlabel("número de segmentos (K)")
ax.set_ylabel("silhueta média")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: virg(v, 2)))
fig.suptitle(f"Com {K} segmentos, a divisão é a mais nítida (maior silhueta) entre as opções úteis para o negócio,\n"
             "e o churn vai de menos de 5% a mais de 50% entre os segmentos", x=0.01, ha="left", fontsize=12, fontweight="bold")
rodape(fig, ["tenure", "Contract", "InternetService", "PaymentMethod"], fonte="gold_churn_treino")
plt.show()

> **Como ler**
> - **Barras:** a silhueta média para cada K. Laranja = o K escolhido; azul = as outras opções entre 4 e 6; cinza = fora
>   da faixa útil (poucos grupos dizem pouco; muitos grupos ficam difíceis de usar numa campanha).
> - **Texto acima da barra:** o menor e o maior churn entre os segmentos. Quanto mais separados, mais útil a segmentação
>   para priorizar ações.
> - **Silhueta de cerca de 0,2:** esperada com variáveis 0/1. Os segmentos são tendências, não caixas fechadas: um
>   cliente na fronteira pode se parecer com dois segmentos.

In [ ]:
# 2.2 Segmentação final: ajusta no treino, aplica na inferência; ordena os segmentos do maior para o menor churn
km = KMeans(n_clusters=K, n_init=10, random_state=42).fit(X_treino)
ordem = y.groupby(km.labels_).mean().sort_values(ascending=False).index          # S1 = maior churn histórico
codigo = {g: f"S{i + 1}" for i, g in enumerate(ordem)}
treino["segmento"] = pd.Series(km.labels_, index=treino.index).map(codigo)
inferencia["segmento"] = pd.Series(km.predict(X_inf), index=inferencia.index).map(codigo)   # mesmos centroides do treino

# nome de negócio para cada segmento, a partir do perfil (independe da numeração interna do K-means)
perfil = X_treino.groupby(treino["segmento"]).mean()
perfil["meses de casa (mediana)"] = treino.groupby("segmento")["tenure"].median()
def nomear(p: pd.Series) -> str:
    if p["sem internet"] > 0.8:
        return "Só telefonia"
    if p["contrato de dois anos"] > 0.8:
        return "Fiéis de dois anos, pacote completo" if p[["segurança online", "suporte técnico"]].mean() > 0.6 else "Contrato de dois anos"
    if p["contrato anual"] > 0.8:
        return "Contrato anual"
    if p["contrato mensal"] > 0.8 and p["internet fibra"] > 0.8:
        return "Fibra no contrato mensal, pouca proteção" if p[["segurança online", "suporte técnico"]].mean() < 0.3 else "Fibra no contrato mensal"
    if p["contrato mensal"] > 0.8 and p["internet DSL"] > 0.8:
        return "DSL no contrato mensal, clientes novos" if p["meses de casa (mediana)"] <= 12 else "DSL no contrato mensal"
    contrato = p[["contrato mensal", "contrato anual", "contrato de dois anos"]].idxmax()
    internet = p[["internet DSL", "internet fibra", "sem internet"]].idxmax()
    return f"{contrato.capitalize()}, {internet}"
NOMES = {s: nomear(perfil.loc[s]) for s in perfil.index}
treino["nome_segmento"] = treino["segmento"].map(NOMES)
inferencia["nome_segmento"] = inferencia["segmento"].map(NOMES)
for s in sorted(NOMES):
    print(f"{s}: {NOMES[s]}")
RESPOSTAS["P3"] = "; ".join(f"{s} {NOMES[s]}" for s in sorted(NOMES))
print("\nResposta P3:", RESPOSTAS["P3"])

## 3. Perfil de cada segmento

> **P3 — Que tipos de cliente temos, e o que caracteriza cada um?**
> **P4 — Onde o churn e a receita se concentram?**

**O que faço:** descrevo cada segmento pelo cliente médio (centroide) e pelo que interessa ao negócio: quantos clientes,
quanto da receita e quanto churn histórico.

In [ ]:
# 3.1 Tabela de perfil: tamanho, churn, receita e as características que definem cada segmento
receita_total = treino["MonthlyCharges"].sum()
resumo = treino.groupby("segmento").agg(clientes=("customerID", "size"), churn_pct=("Churn", "mean"),
                                        mensalidade_media=("MonthlyCharges", "mean"), receita=("MonthlyCharges", "sum"),
                                        meses_de_casa_mediana=("tenure", "median"))
resumo["nome"] = resumo.index.map(NOMES)
resumo["pct_carteira"] = 100 * resumo["clientes"] / len(treino)
resumo["pct_receita"] = 100 * resumo["receita"] / receita_total
resumo["churn_pct"] = 100 * resumo["churn_pct"]
resumo["pct_dos_que_sairam"] = 100 * treino.groupby("segmento")["Churn"].sum() / treino["Churn"].sum()
resumo = resumo[["nome", "clientes", "pct_carteira", "pct_receita", "churn_pct", "pct_dos_que_sairam", "mensalidade_media", "meses_de_casa_mediana"]]
display(resumo.round(1).reset_index())
s1, ult = resumo.iloc[0], resumo.iloc[-1]
RESPOSTAS["P4"] = (f"{resumo.index[0]} ({s1['nome']}) tem {virg(s1['pct_carteira'], 0)}% da carteira, {virg(s1['pct_receita'], 0)}% da receita "
                   f"e {virg(s1['pct_dos_que_sairam'], 0)}% dos que saíram (churn de {virg(s1['churn_pct'], 0)}%); "
                   f"no outro extremo, {resumo.index[-1]} ({ult['nome']}) tem churn de {virg(ult['churn_pct'], 0)}%")
print("Resposta P4:", RESPOSTAS["P4"])

In [ ]:
# 3.2 Mapa do perfil: % de clientes com cada característica, por segmento
mostrar = ["contrato mensal", "contrato anual", "contrato de dois anos", "internet fibra", "internet DSL", "sem internet",
           "cheque eletrônico", "pagamento automático", "fatura digital", "segurança online", "suporte técnico",
           "backup online", "proteção de aparelho", "idoso", "tem família"]
M = 100 * perfil[mostrar].T
fig, ax = plt.subplots(figsize=(12, 7.2))
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list("az", ["#ffffff", COR])
ax.imshow(M.values, cmap=cmap, vmin=0, vmax=100, aspect="auto")
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        v = M.values[i, j]
        ax.text(j, i, f"{virg(v, 0)}%", ha="center", va="center", fontsize=9, color="white" if v > 60 else TEXTO)
ax.set_xticks(range(M.shape[1]), [f"{s}\n{textwrap.fill(NOMES[s], 18)}\nchurn {virg(resumo.loc[s, 'churn_pct'], 0)}% · "
                                  f"{virg(resumo.loc[s, 'clientes'], 0)} clientes" for s in M.columns], fontsize=8.5)
ax.xaxis.tick_top()
ax.set_yticks(range(M.shape[0]), M.index, fontsize=9.5)
ax.tick_params(length=0)
for sp in ax.spines.values():
    sp.set_visible(False)
fig.suptitle("Cada segmento tem um perfil claro: o contrato e o tipo de internet separam os grupos, e a falta de\n"
             "proteção e o cheque eletrônico se concentram nos segmentos de maior churn", x=0.01, ha="left", fontsize=12, fontweight="bold")
rodape(fig, ["Contract", "InternetService", "PaymentMethod", "PaperlessBilling", "OnlineSecurity", "TechSupport",
             "OnlineBackup", "DeviceProtection", "SeniorCitizen", "Partner", "Dependents"], fonte="gold_churn_treino")
plt.show()

> **Como ler o mapa**
> - **Cada coluna é um segmento**, do maior churn (S1, à esquerda) para o menor. No topo, o nome, o churn histórico e o
>   número de clientes.
> - **Cada linha é uma característica**, e o número é o % de clientes do segmento que a têm. Quanto mais escuro, mais
>   típica é a característica no segmento.
> - **O que procurar:** as linhas que mudam muito de uma coluna para outra são as que **definem** os segmentos (contrato
>   e internet). As que mudam pouco (por exemplo, idoso) só complementam o perfil.

In [ ]:
# 3.3 Churn, carteira e receita por segmento
fig, ax = plt.subplots(figsize=(11, 4.2))
r = resumo.iloc[::-1]
yy = np.arange(len(r))
ax.barh(yy, r["churn_pct"], color=[DESTAQUE if v > 30 else COR if v < 15 else CINZA for v in r["churn_pct"]], height=0.6)
for y0, (s, row) in zip(yy, r.iterrows()):
    ax.text(row["churn_pct"] + 0.8, y0, f"churn {virg(row['churn_pct'], 1)}%  ·  {virg(row['pct_carteira'], 0)}% da carteira  ·  "
            f"{virg(row['pct_receita'], 0)}% da receita  ·  {virg(row['pct_dos_que_sairam'], 0)}% dos que saíram",
            va="center", fontsize=9, color=TEXTO, zorder=3,
            bbox=dict(facecolor="white", edgecolor="none", pad=1.5))   # a linha da média passa por trás do texto
ax.axvline(100 * y.mean(), color="#666666", linestyle="--", linewidth=1, zorder=1)
ax.text(100 * y.mean() + 0.5, len(r) - 0.45, f"média {virg(100 * y.mean(), 1)}%", fontsize=8.5, color="#666666")
ax.set_yticks(yy, [f"{s}  {NOMES[s]}" for s in r.index], fontsize=9.5)
ax.set_xlim(0, r["churn_pct"].max() * 2.6)
ax.set_xticks([])
ax.spines[["bottom", "left"]].set_visible(False)
ax.tick_params(axis="y", length=0)
s1 = resumo.iloc[0]
fig.suptitle(f"O segmento {resumo.index[0]} ({NOMES[resumo.index[0]]}) tem {virg(s1['pct_carteira'], 0)}% da carteira, mas "
             f"{virg(s1['pct_dos_que_sairam'], 0)}% dos que saíram\ne {virg(s1['pct_receita'], 0)}% da receita: é onde a retenção mais rende",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
rodape(fig, ["Churn", "MonthlyCharges"], fonte="gold_churn_treino")
plt.show()

## 4. Segmento × risco × valor (inferência)

> **P5 — Entre os clientes que vamos pontuar, quem priorizar?**
> **P6 — Quanto da receita está em risco, e em que segmento?**
> **P7 — Que oferta fazer a cada segmento?**

### Teoria: a matriz de risco × valor

O modelo dá a **probabilidade de sair** de cada cliente; a mensalidade dá o **valor**. Cruzando os dois:

| | Valor alto | Valor baixo |
|---|---|---|
| **Risco alto** | **Salvar já:** vale uma oferta forte | **Ação barata:** e-mail, aviso no app |
| **Risco baixo** | **Fidelizar:** relacionamento, sem incomodar | **Manter:** nenhuma ação |

- **Risco alto:** o modelo do grupo prevê churn (`Churn_Previsto` = 1, corte definido no 06, o que maximiza o F1).
- **Valor alto:** mensalidade acima da mediana do treino.
- **Receita em risco** = a mensalidade dos clientes que o modelo prevê como churn. Não multiplico pela probabilidade,
  porque a do modelo do grupo é inflada pelo balanceamento (média de 41% para 27% de churn real).
  Somada por segmento, mostra **onde** está o dinheiro em jogo.

O **segmento** diz **que oferta** fazer a cada um: a mesma ligação para um cliente de fibra sem suporte e para um de
DSL recém-chegado deveria oferecer coisas diferentes.

In [ ]:
# 4.1 Risco de churn: do modelo do grupo (gold_churn_inferencia_scored) ou, se a tabela não existir, de uma logística provisória
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score

try:
    pont = spark.table(f"{FQ}.gold_churn_inferencia_scored").toPandas().set_index("customerID")
    inferencia["probabilidade"] = inferencia["customerID"].map(pont["Probabilidade_Churn"]).astype(float)
    inferencia["previsto_sim"] = inferencia["customerID"].map(pont["Churn_Previsto"]).astype(int)
    assert inferencia["probabilidade"].notna().all(), "gold_churn_inferencia_scored não cobre todos os clientes da inferência"
    FONTE_PROB = "modelo do grupo (notebook 06, tabela gold_churn_inferencia_scored, corte definido no 06)"
except Exception as erro:
    selecao = spark.table(f"{FQ}.gold_churn_selecao").toPandas()["coluna"].tolist()
    CATS = {c: sorted(treino[c].unique()) for c in selecao if not pd.api.types.is_numeric_dtype(treino[c])}
    def matriz(df):
        Z = df[selecao].copy()
        for c in CATS:
            Z[c] = pd.Categorical(Z[c], categories=CATS[c])
        return pd.get_dummies(Z, columns=list(CATS), drop_first=True, dtype=float)
    modelo = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    p_oof = cross_val_predict(modelo, matriz(treino), y, cv=StratifiedKFold(5, shuffle=True, random_state=42), method="predict_proba")[:, 1]
    inferencia["probabilidade"] = modelo.fit(matriz(treino), y).predict_proba(matriz(inferencia))[:, 1]
    inferencia["previsto_sim"] = (inferencia["probabilidade"] >= 0.5).astype(int)
    FONTE_PROB = f"logística provisória com a seleção da conferência do 05 (AUC {virg(roc_auc_score(y, p_oof), 3)} na validação cruzada; corte 0,5)"
print("Risco de churn:", FONTE_PROB)
print(f"Clientes da inferência previstos como churn: {int(inferencia['previsto_sim'].sum())} "
      f"({virg(100 * inferencia['previsto_sim'].mean(), 1)}%) · probabilidade média: {virg(100 * inferencia['probabilidade'].mean(), 1)}%")


In [ ]:
# 4.2 Matriz de risco × valor por segmento, na inferência
VALOR_ALTO = treino["MonthlyCharges"].median()                      # mediana do treino
QUADRANTES = {(True, True): "Salvar já", (True, False): "Ação barata", (False, True): "Fidelizar", (False, False): "Manter"}
inferencia["risco"] = np.where(inferencia["previsto_sim"] == 1, "alto", "baixo")             # a previsão Sim/Não do modelo
inferencia["valor"] = np.where(inferencia["MonthlyCharges"] >= VALOR_ALTO, "alto", "baixo")
inferencia["quadrante"] = [QUADRANTES[(r == "alto", v == "alto")] for r, v in zip(inferencia["risco"], inferencia["valor"])]
# receita dos clientes previstos como churn (a probabilidade do modelo do grupo é inflada pelo balanceamento,
# então não a multiplico pela mensalidade: somo a mensalidade de quem o modelo marca como Sim)
inferencia["receita_em_risco"] = inferencia["previsto_sim"] * inferencia["MonthlyCharges"]

matriz_q = pd.crosstab(inferencia["segmento"], inferencia["quadrante"]).reindex(columns=list(QUADRANTES.values()), fill_value=0)
por_seg = inferencia.groupby("segmento").agg(clientes=("customerID", "size"), prob_media=("probabilidade", "mean"),
                                             receita_mensal=("MonthlyCharges", "sum"), receita_em_risco=("receita_em_risco", "sum"))
por_seg["nome"] = por_seg.index.map(NOMES)
por_seg["pct_receita_em_risco"] = 100 * por_seg["receita_em_risco"] / por_seg["receita_em_risco"].sum()
tab = por_seg.join(matriz_q)[["nome", "clientes", "prob_media", "receita_mensal", "receita_em_risco", "pct_receita_em_risco"] + list(QUADRANTES.values())]
display(tab.round(2).reset_index())
print(f"Valor alto = mensalidade a partir de R$ {virg(VALOR_ALTO, 2)} (mediana do treino) · risco alto = previsto como churn pelo modelo")
print(f"Receita mensal dos clientes previstos como churn: R$ {virg(inferencia['receita_em_risco'].sum(), 0)} "
      f"de R$ {virg(inferencia['MonthlyCharges'].sum(), 0)} ({virg(100 * inferencia['receita_em_risco'].sum() / inferencia['MonthlyCharges'].sum(), 1)}%)")
n_salvar = int((inferencia["quadrante"] == "Salvar já").sum())
seg_salvar = matriz_q["Salvar já"].idxmax()
RESPOSTAS["P5"] = (f"Os {n_salvar} clientes de \"salvar já\" (risco alto e mensalidade alta); "
                   f"{int(matriz_q.loc[seg_salvar, 'Salvar já'])} deles estão no {seg_salvar} ({NOMES[seg_salvar]})")
top = por_seg["receita_em_risco"].idxmax()
RESPOSTAS["P6"] = (f"R$ {virg(inferencia['receita_em_risco'].sum(), 0)} por mês vêm de clientes que o modelo prevê como churn "
                   f"({virg(100 * inferencia['receita_em_risco'].sum() / inferencia['MonthlyCharges'].sum(), 0)}% da receita); "
                   f"{virg(por_seg.loc[top, 'pct_receita_em_risco'], 0)}% dela no {top} ({NOMES[top]}). "
                   "É a receita de quem tem sinal de saída, e não a perda esperada: como o modelo marca mais clientes "
                   "do que os que saem (precisão de cerca de 52% no teste), pouco mais da metade desse valor deve se perder")
print("\nResposta P5:", RESPOSTAS["P5"])
print("Resposta P6:", RESPOSTAS["P6"])

In [ ]:
# 4.3 Ação sugerida por segmento, a partir do perfil (seção 3) e do perfil de risco do 03
def acoes(s: str) -> list:
    p = perfil.loc[s]
    a = []
    if p["contrato mensal"] > 0.8:
        a.append("oferecer migração para contrato anual, com desconto nos primeiros meses")
    if p["internet fibra"] > 0.8 and p[["segurança online", "suporte técnico"]].mean() < 0.3:
        a.append("incluir suporte técnico ou segurança online por alguns meses")
    if p["cheque eletrônico"] > 0.35:
        a.append("desconto para trocar o cheque eletrônico por débito automático ou cartão")
    if p["meses de casa (mediana)"] <= 12:
        a.append("acompanhamento nos primeiros 6 meses (boas-vindas e contato proativo)")
    if resumo.loc[s, "churn_pct"] < 10:
        a = ["sem ação de retenção: manter o relacionamento e oferecer upgrade quando fizer sentido"]
    elif not a:
        a.append("contato de relacionamento antes da renovação do contrato")
    return a
ACOES = {s: "; ".join(acoes(s)) for s in resumo.index}
plano = pd.DataFrame({"segmento": list(ACOES), "nome": [NOMES[s] for s in ACOES],
                      "churn_historico_%": resumo["churn_pct"].round(1).values,
                      "salvar_ja": [int(matriz_q.loc[s, "Salvar já"]) for s in ACOES],
                      "receita_em_risco_R$": [round(por_seg.loc[s, "receita_em_risco"], 0) for s in ACOES],
                      "acao_sugerida": list(ACOES.values())})
display(plano)
RESPOSTAS["P7"] = " | ".join(f"{s}: {ACOES[s]}" for s in resumo.index)
print("Resposta P7: uma ação por segmento, na tabela acima")
inferencia["acao_sugerida"] = inferencia["segmento"].map(ACOES)
treino["acao_sugerida"] = treino["segmento"].map(ACOES)

In [ ]:
# 4.4 Gráfico: clientes por quadrante e receita em risco, por segmento (inferência)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw={"width_ratios": [1.5, 1]}, sharey=True)
segs = list(resumo.index)[::-1]
yy = np.arange(len(segs))
CORES_Q = {"Salvar já": DESTAQUE, "Ação barata": "#f0b27a", "Fidelizar": COR, "Manter": "#d5dde3"}
esq = np.zeros(len(segs))
for q, cor in CORES_Q.items():
    v = matriz_q.reindex(segs)[q].values
    ax1.barh(yy, v, left=esq, color=cor, height=0.6, label=q)
    for y0, (l, w) in enumerate(zip(esq, v)):
        if w >= 25:
            ax1.text(l + w / 2, y0, str(int(w)), ha="center", va="center", fontsize=8.5,
                     color="white" if q in ("Salvar já", "Fidelizar") else TEXTO)
    esq += v
ax1.set_yticks(yy, [f"{s}  {textwrap.fill(NOMES[s], 28)}" for s in segs], fontsize=9)
ax1.set_title("Clientes da inferência por quadrante de risco × valor", loc="left", fontsize=10.5)
ax1.legend(loc="lower right", frameon=False, fontsize=8.5, ncol=2)
ax1.set_xticks([])
ax1.spines[["bottom", "left"]].set_visible(False)
ax1.tick_params(axis="y", length=0)
rr = por_seg.reindex(segs)["receita_em_risco"]
ax2.barh(yy, rr, color=[DESTAQUE if v == rr.max() else CINZA for v in rr], height=0.6)
for y0, v, pc in zip(yy, rr, por_seg.reindex(segs)["pct_receita_em_risco"]):
    ax2.text(v + rr.max() * 0.02, y0, f"R$ {virg(v, 0)}/mês ({virg(pc, 0)}%)", va="center", fontsize=9, color=TEXTO)
ax2.set_xlim(0, rr.max() * 1.6)
ax2.set_title("Receita mensal dos clientes previstos como churn", loc="left", fontsize=10.5)
ax2.set_xticks([])
ax2.spines[["bottom", "left"]].set_visible(False)
ax2.tick_params(axis="y", length=0)
top = rr.idxmax()
fig.suptitle(f"Na inferência, o segmento {top} ({NOMES[top]}) concentra {virg(por_seg.loc[top, 'pct_receita_em_risco'], 0)}% da receita em risco "
             f"e {int(matriz_q.loc[top, 'Salvar já'])} clientes\npara \"salvar já\": é por onde a campanha de retenção deve começar",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
rodape(fig, ["MonthlyCharges", "Contract", "InternetService"], fonte=f"gold_churn_inferencia; risco: {FONTE_PROB}")
plt.show()

> **Como ler**
> - **À esquerda,** cada barra é um segmento, dividida pelos quadrantes: laranja = salvar já (risco alto e valor alto),
>   laranja-claro = ação barata, azul = fidelizar, cinza-claro = manter.
> - **À direita,** a receita mensal dos clientes que o modelo prevê como churn, em cada segmento.
> - **O que procurar:** o segmento com mais laranja e mais receita em risco é a prioridade. Os segmentos quase todos em
>   azul e cinza não precisam de retenção: precisam de relacionamento.
> - **De onde vem o risco:** do modelo do grupo (`gold_churn_inferencia_scored`, corte definido no 06). Uso a previsão Sim/Não
>   e não a probabilidade × mensalidade, porque a probabilidade desse modelo é inflada pelo balanceamento (média de cerca
>   de 41% para 27% de churn real).

## Respostas às perguntas de negócio

A tabela junta as respostas de cada etapa, escritas a partir dos números deste notebook. É o resumo para levar à
apresentação.

In [ ]:
PERGUNTAS = {"P1": "Com que informação descrevemos o cliente, sem usar o churn?",
             "P2": "Quantos grupos de clientes fazem sentido para o negócio usar?",
             "P3": "Que tipos de cliente temos, e o que caracteriza cada um?",
             "P4": "Onde o churn e a receita se concentram?",
             "P5": "Entre os clientes que vamos pontuar, quem priorizar?",
             "P6": "Quanto da receita está em risco, e em que segmento?",
             "P7": "Que oferta fazer a cada segmento?"}
assert set(RESPOSTAS) == set(PERGUNTAS), "falta rodar alguma etapa antes desta célula"
respostas = pd.DataFrame({"pergunta": PERGUNTAS, "resposta": RESPOSTAS}).rename_axis("#").reset_index()
display(respostas)
print("Probabilidade usada nas respostas P5 e P6:", FONTE_PROB)

## 5. Gravar a segmentação

**O que faço:** gravo uma linha por cliente (treino e inferência) com o segmento, o nome, a ação sugerida e, na
inferência, a probabilidade, o quadrante e a receita em risco. E uma tabela com o perfil de cada segmento.

In [ ]:
# 5.1 Grava gold_churn_segmentos (um cliente por linha) e gold_churn_segmentos_perfil (um segmento por linha)
cols = ["customerID", "segmento", "nome_segmento", "acao_sugerida", "MonthlyCharges"]
seg_inf = inferencia[cols + ["probabilidade", "risco", "valor", "quadrante", "receita_em_risco"]].assign(base="inferencia")
seg_tr = treino[cols + ["Churn"]].assign(base="treino")
segmentos = pd.concat([seg_tr, seg_inf], ignore_index=True)
segmentos["fonte_probabilidade"] = np.where(segmentos["base"] == "inferencia", FONTE_PROB, None)
assert segmentos["customerID"].is_unique and len(segmentos) == len(treino) + len(inferencia)

perfil_seg = resumo.join(perfil[mostrar].mul(100).round(1)).reset_index()
perfil_seg["acao_sugerida"] = perfil_seg["segmento"].map(ACOES)
# o Delta não aceita espaço nem acento confuso no nome da coluna: "contrato mensal" vira "pct_contrato_mensal"
import unicodedata
def nome_coluna(c):
    if c in resumo.reset_index().columns or c == "acao_sugerida":
        return c
    sem_acento = unicodedata.normalize("NFKD", c).encode("ascii", "ignore").decode()
    return "pct_" + sem_acento.replace(" ", "_")
perfil_seg.columns = [nome_coluna(c) for c in perfil_seg.columns]
for nome, df, desc in [("gold_churn_segmentos", segmentos, "Segmento de cada cliente (K-means no comportamento), ação sugerida e, na inferência, risco x valor"),
                       ("gold_churn_segmentos_perfil", perfil_seg, "Perfil de cada segmento: tamanho, churn histórico, receita e % de clientes com cada característica")]:
    (spark.createDataFrame(df).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ}.{nome}"))
    spark.sql(f"COMMENT ON TABLE {FQ}.{nome} IS '{desc}'")
    print(f"{FQ}.{nome}: {virg(len(df), 0)} linhas")

## Resumo e checklist final do 07

| # | Decisão | Status | Onde |
|---|---|---|---|
| D20 | Segmentação de clientes: K-means no comportamento (5 segmentos), sem o churn, cruzado com risco × valor | ◐ Sugestão para o negócio e a apresentação (11/11) | Seções 2 a 4 |

**Limites a ter em mente:**
- **Silhueta de cerca de 0,2:** os segmentos são tendências claras de perfil, não grupos totalmente separados.
- **O risco vem do modelo do grupo:** previsão Sim/Não no corte definido no 06 (maior F1). A receita em risco é a receita dos
  clientes previstos como churn, não uma perda esperada.
- **Segmento não é causa:** o S1 tem mais churn **e** um perfil típico; a ação sugerida é uma hipótese de negócio a
  testar (por exemplo, oferecer a migração de contrato para metade do segmento e comparar com a outra metade).

**Saída do 07:** `gold_churn_segmentos` (7.043 clientes, um por linha) e `gold_churn_segmentos_perfil` (5 segmentos).

**Ordem de execução:** 05 → 06 (grava `gold_churn_inferencia_scored`) → este notebook → 08 (entrega).